# Inspect Solver Output (100K)

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = "../data/solver_output_100k"

meta = np.load(f"{DATA_DIR}/meta.npy")
ranges = np.load(f"{DATA_DIR}/ranges.npy")
values = np.load(f"{DATA_DIR}/values.npy")

print(f"meta:   shape={meta.shape}, dtype={meta.dtype}")
print(f"ranges: shape={ranges.shape}, dtype={ranges.dtype}")
print(f"values: shape={values.shape}, dtype={values.dtype}")

meta:   shape=(100000, 6), dtype=float32
ranges: shape=(100000, 2652), dtype=float32
values: shape=(100000, 2652), dtype=float32


## Data Layout

**meta.npy** — `[N, 6]`

| Column | Description |
|--------|-------------|
| 0 | Flop card 0 (0–51) |
| 1 | Flop card 1 (0–51) |
| 2 | Flop card 2 (0–51) |
| 3 | Turn card (0–51) |
| 4 | Pot size |
| 5 | Effective stack |

**ranges.npy** — `[N, 2652]`: `reach_oop[1326] ++ reach_ip[1326]`

**values.npy** — `[N, 2652]`: `cfv_oop[1326] ++ cfv_ip[1326]` (pot-normalized)

In [2]:
# Meta as DataFrame
meta_df = pd.DataFrame(meta, columns=["flop_0", "flop_1", "flop_2", "turn", "pot", "stack"])
meta_df = meta_df.astype({"flop_0": int, "flop_1": int, "flop_2": int, "turn": int})
meta_df.head(10)

,flop_0,flop_1,flop_2,turn,pot,stack
0,22,29,45,35,5100.0,4600.0
1,0,10,34,6,4300.0,3900.0
2,12,13,18,41,2700.0,3400.0
3,35,38,44,25,2700.0,6900.0
4,3,29,48,18,1700.0,8100.0
5,3,34,45,42,800.0,2200.0
6,11,13,37,49,5300.0,2600.0
7,32,43,46,17,1100.0,5200.0
8,2,30,31,50,4900.0,6900.0
9,10,15,28,26,3600.0,1300.0


In [ ]:
# Card index → human-readable
RANKS = "23456789TJQKA"
SUITS = "cdhs"

def card_str(idx):
    idx = int(idx)
    return RANKS[idx // 4] + SUITS[idx % 4]

meta_df["board"] = meta_df.apply(
    lambda r: f"{card_str(r.flop_0)} {card_str(r.flop_1)} {card_str(r.flop_2)} | {card_str(r.turn)}", axis=1
)
meta_df[["board", "pot", "stack"]].head(10)

In [ ]:
# Meta summary statistics
meta_df[["pot", "stack"]].describe()

In [ ]:
# Unique boards
unique_boards = meta_df[["flop_0", "flop_1", "flop_2", "turn"]].drop_duplicates()
unique_flops = meta_df[["flop_0", "flop_1", "flop_2"]].drop_duplicates()
print(f"Total samples:  {len(meta_df):,}")
print(f"Unique boards (flop+turn): {len(unique_boards):,}")
print(f"Unique flops:  {len(unique_flops):,}")
print(f"Avg samples per board: {len(meta_df) / len(unique_boards):.1f}")
print(f"Avg turn cards per flop: {len(unique_boards) / len(unique_flops):.1f}")

In [ ]:
# Ranges head — OOP (first 10 combos)
oop_range = ranges[:, :1326]
ip_range = ranges[:, 1326:]

print("OOP reaches (first 5 samples, first 10 combos):")
pd.DataFrame(oop_range[:5, :10], columns=[f"combo_{i}" for i in range(10)])

In [ ]:
# Ranges summary
print("=== OOP Reaches ===")
print(f"  min={oop_range.min():.6f}, max={oop_range.max():.6f}")
print(f"  sum per sample: mean={oop_range.sum(axis=1).mean():.4f}, std={oop_range.sum(axis=1).std():.4f}")
print(f"  nonzero per sample: mean={(oop_range > 0).sum(axis=1).mean():.0f}")
print()
print("=== IP Reaches ===")
print(f"  min={ip_range.min():.6f}, max={ip_range.max():.6f}")
print(f"  sum per sample: mean={ip_range.sum(axis=1).mean():.4f}, std={ip_range.sum(axis=1).std():.4f}")
print(f"  nonzero per sample: mean={(ip_range > 0).sum(axis=1).mean():.0f}")

In [ ]:
# Values head — OOP CFVs (first 10 combos)
oop_cfv = values[:, :1326]
ip_cfv = values[:, 1326:]

print("OOP CFVs (first 5 samples, first 10 combos):")
pd.DataFrame(oop_cfv[:5, :10], columns=[f"combo_{i}" for i in range(10)])

In [ ]:
# Values summary
print("=== OOP CFVs (pot-normalized) ===")
print(f"  min={oop_cfv.min():.4f}, max={oop_cfv.max():.4f}, mean={oop_cfv.mean():.4f}")
print()
print("=== IP CFVs (pot-normalized) ===")
print(f"  min={ip_cfv.min():.4f}, max={ip_cfv.max():.4f}, mean={ip_cfv.mean():.4f}")
print()
# Zero-sum check: reach-weighted OOP + IP should ≈ 0
weighted_oop = (oop_range * oop_cfv).sum(axis=1)
weighted_ip = (ip_range * ip_cfv).sum(axis=1)
total = weighted_oop + weighted_ip
print("=== Zero-sum check (reach-weighted OOP + IP) ===")
print(f"  mean={total.mean():.6f}, std={total.std():.6f}, max_abs={np.abs(total).max():.6f}")